# Crypto.Net · 05 — Polkadot and Substrate

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./05-polkadot.id.ipynb)

sr25519 and ed25519 accounts with Substrate derivation paths, SS58 addresses, message signatures, SCALE, and an extrinsic built from live runtime metadata whose fee is quoted by the chain.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Polkadot;

$"sr25519 needs the Rust core — loaded: {NativeLoader.IsNativeAvailable}"

## Development accounts

`//Alice` is derived from Substrate's public development phrase — the same account every local node pre-funds.

In [ ]:
var alice   = PolkadotAccount.FromSuri("//Alice", chain: PolkadotChain.Westend);
var aliceEd = PolkadotAccount.FromSuri("//Alice", SignatureScheme.Ed25519, PolkadotChain.Westend);

new[] { PolkadotChain.Polkadot, PolkadotChain.Kusama, PolkadotChain.Westend }
    .Select(c => new { Network = c.Name, c.Ss58Prefix, Sr25519 = alice.AddressFor(c), Ed25519 = aliceEd.AddressFor(c) })

## Your accounts

The empty path is the root account Polkadot.js, Talisman and SubWallet show for a phrase; `//hard` and `/soft` junctions derive more.

In [ ]:
var wallet = HdWallet.Generate();
var root   = wallet.GetPolkadotAccount("", PolkadotChain.Polkadot);
var child  = wallet.GetPolkadotAccount("//polkadot//0", PolkadotChain.Polkadot);
var (prefix, publicKey) = Ss58Address.Decode(root.Address.Value);
Console.WriteLine($"root  {root.Address}  (prefix {prefix}, key {HexUtil.Encode(publicKey)})");
Console.WriteLine($"child {child.Address}  ({child.DerivationPath})");

## Signing messages

`SignMessage` wraps the text in `<Bytes>…</Bytes>` like the Polkadot.js extension's `signRaw`. sr25519 signatures are randomized, so each run differs, but they always verify.

In [ ]:
byte[] sig = root.SignMessage("Hello Polkadot");
Console.WriteLine(HexUtil.Encode(sig));
Console.WriteLine($"valid: {PolkadotAccount.VerifyMessage("Hello Polkadot", sig, root.PublicKey.Span)}");
Console.WriteLine($"tampered: {PolkadotAccount.VerifyMessage("Hello Kusama", sig, root.PublicKey.Span)}");

## SCALE codec

In [ ]:
var encoded = new ScaleWriter().Compact(1_000_000_000_000).U32(42).String("Crypto.Net").ToArray();
{
    // ScaleReader is a ref struct, so it lives inside a block rather than as a notebook-level variable.
    var reader = new ScaleReader(encoded);
    Console.WriteLine($"{HexUtil.Encode(encoded)} → {reader.Compact()}, {reader.U32()}, {reader.String()}");
}

## Live runtime metadata (Westend Asset Hub)

Pallet indices, call indices and the signed-extension list are read from the chain, so extrinsics keep working after runtime upgrades.

In [ ]:
var rpc = new PolkadotRpcClient(PolkadotChain.WestendAssetHub);
var (spec, txVersion, specName) = await rpc.GetRuntimeVersionAsync();
var metadata = await rpc.GetMetadataAsync();
var (pallet, callIndex) = metadata.GetCallIndex("Balances", "transfer_keep_alive");

Console.WriteLine($"{await rpc.GetChainNameAsync()} · {specName} v{spec} · metadata V{metadata.Version} · {metadata.Pallets.Count} pallets");
Console.WriteLine($"Balances.transfer_keep_alive = ({pallet}, {callIndex})");
Console.WriteLine($"signed extensions: {string.Join(", ", metadata.SignedExtensions.Select(e => e.Identifier))}");
var aliceInfo = await rpc.GetAccountInfoAsync(alice.AddressFor(PolkadotChain.WestendAssetHub));
Console.WriteLine($"//Alice on Westend Asset Hub: nonce {aliceInfo.Nonce}, free {new Amount(aliceInfo.Free, 12)} WND");

## A signed extrinsic, priced by the chain

The runtime must decode an extrinsic to quote its fee, so a fee estimate proves the encoding is valid. Nothing is submitted.

In [ ]:
var ctx = await rpc.GetExtrinsicContextAsync(alice.Address.Value);
byte[] call = Extrinsic.TransferKeepAlive(metadata, root.AddressFor(PolkadotChain.WestendAssetHub), 1_000_000_000_000);
byte[] extrinsic = Extrinsic.Sign(metadata, call, alice, ctx);

Console.WriteLine($"{extrinsic.Length} bytes, hash {Extrinsic.Hash(extrinsic)}");
Console.WriteLine($"mortal for {ctx.MortalPeriod} blocks from #{ctx.BlockNumber:N0}, nonce {ctx.Nonce}");
Console.WriteLine($"fee quoted by the runtime: {await rpc.EstimateFeeAsync(extrinsic)} WND");

## Sending for real

```csharp
TxHash hash = await rpc.TransferAsync(root, "5Grw…", Amount.Parse("0.5", 12));
TxHash remark = await rpc.SubmitCallAsync(root, Extrinsic.Remark(metadata, "hello"u8));
```

In [ ]:
foreach (IDisposable d in new IDisposable[] { alice, aliceEd, root, child, rpc, wallet }) d.Dispose();
"disposed"